# 📘 Fincept Notebook — Mean-Variance Optimization

**Portfolio · Hard · ~35 min · pandas + numpy**

Harry Markowitz's mean-variance framework is the foundation of modern portfolio theory. The core idea: a portfolio is not just a basket of returns — it is a trade-off between *expected return* and *risk* (volatility), and because assets are imperfectly correlated, the risk of the whole is less than the sum of its parts. In this notebook we build a covariance matrix from volatilities and correlations, simulate thousands of long-only portfolios, and locate the two portfolios every quant cares about: the **maximum-Sharpe** (tangency) portfolio and the **minimum-variance** portfolio.

**What you'll learn**
- How to build an annualized covariance matrix from volatilities + a correlation matrix
- How to compute portfolio expected return, volatility, and Sharpe ratio from weights
- How Monte Carlo sampling traces out the efficient frontier and reveals the diversification benefit

> **Requires** `pandas` and `numpy` (bundled with Fincept Terminal's Python environment).


## 1. The inputs: expected returns, vols, and correlations

Mean-variance optimization needs three things for a universe of *N* assets:

1. **Expected annual returns** `μ` — a length-N vector.
2. **Annual volatilities** `σ` — a length-N vector (the standard deviation of each asset's annual return).
3. **A correlation matrix** `R` — how the assets move together.

The **covariance matrix** is then built as `Σ = D · R · D`, where `D = diag(σ)`. Element `Σ[i,j] = σ_i · σ_j · ρ_ij`. We use four realistic assets: US Equity, Intl Equity, Bonds, and Gold.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

assets = ["US_Equity", "Intl_Equity", "Bonds", "Gold"]

# Expected ANNUAL returns and ANNUAL volatilities (realistic long-run estimates)
mu  = np.array([0.085, 0.075, 0.035, 0.055])   # expected annual return
vol = np.array([0.160, 0.180, 0.060, 0.150])   # annual volatility (std dev)

# Correlation matrix (symmetric, 1s on diagonal)
corr = np.array([
    [1.00, 0.80, 0.10, 0.15],
    [0.80, 1.00, 0.05, 0.20],
    [0.10, 0.05, 1.00, 0.25],
    [0.15, 0.20, 0.25, 1.00],
])

# Covariance = diag(vol) @ corr @ diag(vol)
D = np.diag(vol)
cov = D @ corr @ D

print("Expected annual return & volatility per asset")
print(pd.DataFrame({"E[return]": mu, "volatility": vol}, index=assets).round(4).to_string())
print()
print("Correlation matrix")
print(pd.DataFrame(corr, index=assets, columns=assets).round(2).to_string())
print()
print("Annualized covariance matrix (Σ = D·R·D)")
print(pd.DataFrame(cov, index=assets, columns=assets).round(5).to_string())
print()
print("Sanity check: sqrt(diag(cov)) should equal the input vols ->", np.allclose(np.sqrt(np.diag(cov)), vol))

## 2. Portfolio math: return, volatility, Sharpe

Given a weight vector `w` (long-only, `w_i ≥ 0`, `Σ w_i = 1`):

- **Expected return:** `μ_p = wᵀ μ`
- **Variance:** `σ²_p = wᵀ Σ w` &nbsp;→&nbsp; **Volatility:** `σ_p = √(wᵀ Σ w)`
- **Sharpe ratio:** `(μ_p − r_f) / σ_p`, with risk-free rate `r_f`.

The `wᵀ Σ w` term is the magic: cross-terms `w_i w_j σ_ij` with low correlation *reduce* total variance below the weighted average of individual variances. That is diversification, quantified. Let's verify on an equal-weight portfolio.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

assets = ["US_Equity", "Intl_Equity", "Bonds", "Gold"]
mu  = np.array([0.085, 0.075, 0.035, 0.055])
vol = np.array([0.160, 0.180, 0.060, 0.150])
corr = np.array([
    [1.00, 0.80, 0.10, 0.15],
    [0.80, 1.00, 0.05, 0.20],
    [0.10, 0.05, 1.00, 0.25],
    [0.15, 0.20, 0.25, 1.00],
])
D = np.diag(vol)
cov = D @ corr @ D
rf = 0.03   # risk-free rate

def port_return(w):     return float(w @ mu)
def port_vol(w):        return float(np.sqrt(w @ cov @ w))
def port_sharpe(w):     return (port_return(w) - rf) / port_vol(w)

# Equal-weight portfolio
w_eq = np.repeat(1.0 / len(assets), len(assets))

# Weighted-average vol (what you'd get with NO diversification, i.e. perfect correlation)
weighted_avg_vol = float(w_eq @ vol)

print("Equal-weight portfolio (25% each)")
print(f"  Expected return : {port_return(w_eq):.4f}")
print(f"  Volatility      : {port_vol(w_eq):.4f}")
print(f"  Sharpe ratio    : {port_sharpe(w_eq):.4f}")
print()
print(f"  Weighted-avg vol (no diversification) : {weighted_avg_vol:.4f}")
print(f"  Actual portfolio vol                  : {port_vol(w_eq):.4f}")
print(f"  Diversification benefit (vol saved)   : {weighted_avg_vol - port_vol(w_eq):.4f}")

## 3. Monte Carlo: simulate 10,000 random portfolios

There is a closed-form solution for the efficient frontier, but a Monte Carlo sweep is more intuitive and handles the long-only constraint naturally. We draw 10,000 weight vectors from a **Dirichlet distribution**, which produces non-negative weights that sum to 1 by construction — exactly the long-only simplex. For each portfolio we record return, volatility, and Sharpe. The cloud of points fills the *feasible region*; its upper-left boundary is the efficient frontier.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

assets = ["US_Equity", "Intl_Equity", "Bonds", "Gold"]
mu  = np.array([0.085, 0.075, 0.035, 0.055])
vol = np.array([0.160, 0.180, 0.060, 0.150])
corr = np.array([
    [1.00, 0.80, 0.10, 0.15],
    [0.80, 1.00, 0.05, 0.20],
    [0.10, 0.05, 1.00, 0.25],
    [0.15, 0.20, 0.25, 1.00],
])
cov = np.diag(vol) @ corr @ np.diag(vol)
rf = 0.03

N_PORT = 10000
n = len(assets)

# Dirichlet(1,...,1) = uniform over the long-only simplex (weights >=0, sum to 1)
W = np.random.dirichlet(np.ones(n), size=N_PORT)   # shape (N_PORT, n)

rets = W @ mu                                        # expected returns, shape (N_PORT,)
vols = np.sqrt(np.einsum('ij,jk,ik->i', W, cov, W)) # portfolio vols, vectorized wᵀΣw
sharpes = (rets - rf) / vols

sim = pd.DataFrame({"return": rets, "vol": vols, "sharpe": sharpes})

print(f"Simulated {N_PORT:,} long-only portfolios (Dirichlet weights, seed=7)")
print()
print("Summary statistics across the cloud:")
print(sim.describe().round(4).to_string())
print()
print("First 5 sampled portfolios:")
head = sim.head(5).copy()
for j, a in enumerate(assets):
    head[a] = W[:5, j]
print(head.round(4).to_string())

## 4. The two special portfolios

From the cloud we extract:

- **Maximum-Sharpe (tangency) portfolio** — the one with the highest risk-adjusted return `(μ_p − r_f)/σ_p`. This is the portfolio a rational investor combines with the risk-free asset.
- **Minimum-volatility portfolio** — the leftmost point of the frontier; the lowest-risk mix achievable, regardless of return.

Note that Monte Carlo only *approximates* the true optima — with more samples the estimates tighten. We print full weight vectors for both.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

assets = ["US_Equity", "Intl_Equity", "Bonds", "Gold"]
mu  = np.array([0.085, 0.075, 0.035, 0.055])
vol = np.array([0.160, 0.180, 0.060, 0.150])
corr = np.array([
    [1.00, 0.80, 0.10, 0.15],
    [0.80, 1.00, 0.05, 0.20],
    [0.10, 0.05, 1.00, 0.25],
    [0.15, 0.20, 0.25, 1.00],
])
cov = np.diag(vol) @ corr @ np.diag(vol)
rf = 0.03

N_PORT = 10000
n = len(assets)
W = np.random.dirichlet(np.ones(n), size=N_PORT)
rets = W @ mu
vols = np.sqrt(np.einsum('ij,jk,ik->i', W, cov, W))
sharpes = (rets - rf) / vols

i_sharpe = int(np.argmax(sharpes))     # max-Sharpe portfolio index
i_minvol = int(np.argmin(vols))        # min-volatility portfolio index

def describe(i, label):
    w = W[i]
    print(label)
    print(f"  Expected return : {rets[i]:.4f}")
    print(f"  Volatility      : {vols[i]:.4f}")
    print(f"  Sharpe ratio    : {sharpes[i]:.4f}")
    print("  Weights:")
    for a, wi in zip(assets, w):
        print(f"    {a:<12} {wi:7.2%}")
    print()

describe(i_sharpe, "=== MAXIMUM-SHARPE (tangency) PORTFOLIO ===")
describe(i_minvol, "=== MINIMUM-VOLATILITY PORTFOLIO ===")

comp = pd.DataFrame({
    "Max-Sharpe": W[i_sharpe],
    "Min-Vol":    W[i_minvol],
}, index=assets)
print("Weight comparison")
print(comp.round(4).to_string())

## 5. Tracing the efficient frontier

The **efficient frontier** is the set of portfolios offering the highest expected return for each level of risk. To approximate it from our cloud, we bucket the simulated portfolios by volatility, and within each bucket keep the one with the maximum return. The result is a monotone, concave curve — the upper edge of the feasible region. Anything below the frontier is dominated (you could get more return for the same risk).


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

assets = ["US_Equity", "Intl_Equity", "Bonds", "Gold"]
mu  = np.array([0.085, 0.075, 0.035, 0.055])
vol = np.array([0.160, 0.180, 0.060, 0.150])
corr = np.array([
    [1.00, 0.80, 0.10, 0.15],
    [0.80, 1.00, 0.05, 0.20],
    [0.10, 0.05, 1.00, 0.25],
    [0.15, 0.20, 0.25, 1.00],
])
cov = np.diag(vol) @ corr @ np.diag(vol)
rf = 0.03

N_PORT = 10000
n = len(assets)
W = np.random.dirichlet(np.ones(n), size=N_PORT)
rets = W @ mu
vols = np.sqrt(np.einsum('ij,jk,ik->i', W, cov, W))
sharpes = (rets - rf) / vols

sim = pd.DataFrame({"return": rets, "vol": vols, "sharpe": sharpes})

# Bucket by volatility, take the max-return portfolio in each bucket -> frontier
n_buckets = 12
sim["vol_bucket"] = pd.cut(sim["vol"], bins=n_buckets)
idx = sim.groupby("vol_bucket", observed=True)["return"].idxmax().dropna().astype(int)
frontier = sim.loc[idx, ["vol", "return", "sharpe"]].sort_values("vol").reset_index(drop=True)

print("Approximate efficient frontier (max return per volatility bucket)")
print(frontier.round(4).to_string())
print()
print("Notice: as volatility rises, the best achievable return rises but with")
print("diminishing slope (concavity) — and Sharpe peaks near the tangency point.")

## 6. Why diversification works

Look back at the min-vol portfolio: it loads heavily on bonds (low vol) but still holds some equity and gold. Why not 100% bonds? Because gold and equities have **low correlation** with bonds — adding a little of each lowers total variance via the negative-to-low cross-terms, even though each is individually riskier. The min-vol portfolio's volatility is typically *below* the lowest single-asset vol weighted average. That free lunch — lower risk for the same or higher return — is the entire point of Markowitz.

**Caveats for real use:** expected returns are notoriously hard to estimate and optimizers are extremely sensitive to them; covariance is more stable but still noisy. Practitioners add constraints (position limits), shrinkage estimators, and robust/Black-Litterman methods. But the geometry you just simulated — the frontier, the tangency portfolio, the min-vol point — is universal.


---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*
